# E5 tuning, Phase A: which graph settings are best? (Kaggle, CPU)

Scores graph settings on **validation data only**, with a fast linear stand-in for A3T-GCN
(`tuning/e5_proxy.py`; no A3T-GCN training, no random seed). Nothing is recomputed from the causal
pipeline: it reuses the VAR coefficients, the LiNGAM B0 and the PEM result you already have.

What it answers, in order:
1. **Calibration**: does the proxy rank the graphs we already trained (NOGRAPH, E0, E1, E2, E3, E5) in the
   same order as A3T-GCN did? If it prints **FAIL**, the proxy cannot be used and the other tables must not be trusted.
2. **K sweep**: how many sources per sensor (5, 10, 15, 25, 40, 60, 100)?
3. **Roles**: what do LiNGAM's B0 and PEM's P add on top of the VAR? (B0~ = 0, B0 alone, P*B0, thresholded P, B0 cut-offs.)
4. **Extras**: any other PEM / LiNGAM results you add (different nu0, nu1, eta, rows) scored the same way.

**Before running** (Add Input)
1. The **GBA dataset** (`gba_his_raw_2021.h5`, `gba_rn_adj.npy`, `gba_meta.csv`).
2. The **PEM inputs dataset** that pem-e7 used: `coefficients.npy`, `sensors.npy`, `lingam_result.npz`
   (and `residuals.npy`, `metadata.json`, which are not needed here).
3. **Add to that dataset** (new files, from your laptop):
   - `pem_result_base.npz`: the E4 PEM result, copy of
     `outputs/pem/lag3_alpha1e+06__nu0_0.0017_nu1_1_tau_0.01_thr_0.5_eta_0.5_L-BFGS-B_std/pem_result.npz` **renamed**.
   - optional `pem_result_E7_road.npz`: the road-prior PEM result (`outputs/e7_output/pem_result.npz`, renamed).
   - for Phase B, one file per extra run, named `pem_result_<settings>.npz` / `lingam_result_<settings>.npz`.
4. Optional: the **artifacts dataset** (its `E5__h*.npz` files are used to verify the E5 graph is rebuilt exactly).
5. Settings: **Accelerator = None (CPU)**, **Internet = On** (the code is cloned from GitHub).
6. **Save Version -> Save & Run All (Commit)**. Expected time: about 1 hour.

Download `e5_tuning_output/` from the *Output* tab (tables, `results.json`, figures).

## Settings

In [ ]:
import os

REPO_URL    = "https://github.com/prof-sanjay/A3TGCN-VAR-PEM-Implementation.git"
REPO_BRANCH = "main"

HORIZONS = [1, 6, 12]                      # 5, 30, 60 minutes
K_LIST   = [5, 10, 15, 25, 40, 60, 100]    # sources kept per sensor in the K sweep (25 = E5)
STRIDE_TRAIN, STRIDE_VAL = 4, 2            # use every 4th training / 2nd validation window

## 1. Environment

In [ ]:
import sys, numpy as np, scipy, pandas as pd
print("Python:", sys.version)
print("NumPy:", np.__version__, "| SciPy:", scipy.__version__, "| pandas:", pd.__version__)
print("CPUs:", os.cpu_count())

## 2. Code, dataset and inputs

In [ ]:
import glob, json, shutil, subprocess

WORK = "/kaggle/working"
CODE = f"{WORK}/implementation"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
for needed in ("tune_e5_proxy.py", "tuning/e5_proxy.py", "tuning/reference_a3tgcn.json"):
    assert os.path.exists(f"{CODE}/{needed}"), f"{needed} is not on GitHub yet: push the tuning files first."

hits = glob.glob("/kaggle/input/**/gba_his_raw_2021.h5", recursive=True)
assert hits, "Add the GBA dataset (Add Input)"
os.makedirs(f"{CODE}/datasets", exist_ok=True)
os.symlink(os.path.dirname(hits[0]), f"{CODE}/datasets/LargeST GBA")


def inputs(pattern):
    return sorted(p for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
                  if "/units/" not in p and "/implementation/" not in p)


def find_one(name):
    found = inputs(name)
    assert len(found) == 1, f"expected exactly one {name} under /kaggle/input, found {found}"
    return found[0]


VAR_DIR = f"{CODE}/outputs/var_train/lag3_alpha1e+06"
os.makedirs(VAR_DIR, exist_ok=True)
for name in ("coefficients.npy", "sensors.npy"):
    os.symlink(find_one(name), f"{VAR_DIR}/{name}")

LINGAM_BASE   = find_one("lingam_result.npz")
LINGAM_EXTRAS = [p for p in inputs("lingam_result_*.npz")]

pem_files = inputs("pem_result*.npz")
def prior_of(path):
    with np.load(path) as r:
        return str(r["prior"]) if "prior" in r.files else "uniform"

uniform = [p for p in pem_files if prior_of(p) == "uniform"]
base = [p for p in uniform if os.path.basename(p) == "pem_result_base.npz"] or (uniform if len(uniform) == 1 else [])
assert len(base) == 1, ("name the E4 PEM result pem_result_base.npz in the dataset; uniform-prior files found: "
                        f"{uniform}")
PEM_BASE   = base[0]
PEM_EXTRAS = [p for p in pem_files if p != PEM_BASE]

GRAPHS = f"{WORK}/graphs_for_check"
os.makedirs(GRAPHS, exist_ok=True)
for p in glob.glob("/kaggle/input/**/E5__h*.npz", recursive=True):
    shutil.copy(p, f"{GRAPHS}/{os.path.basename(p).replace('alpha1e06', 'alpha1e+06')}")

print("LiNGAM base   :", LINGAM_BASE)
print("LiNGAM extras :", LINGAM_EXTRAS)
print("PEM base      :", PEM_BASE)
print("PEM extras    :", [(p, prior_of(p)) for p in PEM_EXTRAS])
print("E5 graphs for the reconstruction check:", sorted(os.listdir(GRAPHS)))

## 3. Run the tuning stages

Writes `results.json` and one CSV per stage after each stage, so a stopped run keeps what it finished.

In [ ]:
import time

OUT = f"{WORK}/e5_tuning_output"
os.makedirs(OUT, exist_ok=True)

cmd = [sys.executable, "-u", "tune_e5_proxy.py",
       "--var-dir", VAR_DIR, "--lingam-result", LINGAM_BASE, "--pem-base", PEM_BASE,
       "--out", OUT, "--horizons", *map(str, HORIZONS), "--k-list", *map(str, K_LIST),
       "--stride-train", str(STRIDE_TRAIN), "--stride-val", str(STRIDE_VAL)]
if PEM_EXTRAS:
    cmd += ["--pem-extra", *PEM_EXTRAS]
if LINGAM_EXTRAS:
    cmd += ["--lingam-extra", *LINGAM_EXTRAS]
if os.listdir(GRAPHS):
    cmd += ["--graphs-dir", GRAPHS]
print(" ".join(cmd))

start = time.time()
log = open(f"{OUT}/tuning.log", "w")
proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    log.write(line); log.flush()
    print(line.rstrip(), flush=True)
proc.wait()
log.close()
print(f"exit code {proc.returncode} | {(time.time() - start) / 60:.1f} min")
assert proc.returncode == 0

## 4. Results

In [ ]:
res = json.load(open(f"{OUT}/results.json"))
pd.set_option("display.width", 220, "display.max_columns", 30)

if "e5_reconstruction_check" in res:
    print("E5 graph rebuilt here vs the stored artifact:", json.dumps(res["e5_reconstruction_check"], indent=1))

cal = res.get("calibration")
if cal:
    print(f"\nCALIBRATION: mean Spearman {cal['mean_spearman']:.2f} (needs >= {cal['threshold']}) -> "
          f"{'PASS' if cal['pass'] else 'FAIL: do not use the tables below'}")
    for h, c in cal["per_horizon"].items():
        print(f"  {h}: rho = {c['spearman']:.2f} over {c['graphs']}")
    print(pd.read_csv(f"{OUT}/calibration_vs_a3tgcn.csv").to_string(index=False))

In [ ]:
for name, title in (("k_sweep", "K sweep: top-K sources per sensor (lower proxy RMSE = better)"),
                    ("roles", "What do B0 and P add? (K = 25)"),
                    ("extras", "Extra PEM / LiNGAM results (K = 25)")):
    path = f"{OUT}/{name}.csv"
    if os.path.exists(path):
        print("\n" + title)
        print(pd.read_csv(path).to_string(index=False))
print("\nE5 as built (K = 25, P * B0):")
print(pd.Series(res["base_E5"]).to_string())

## 5. Figure

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
if os.path.exists(f"{OUT}/k_sweep.csv"):
    k = pd.read_csv(f"{OUT}/k_sweep.csv")
    for h in HORIZONS:
        axes[0].plot(k["K"], k[f"val_rmse_h{h}"], marker="o", label=f"h = {h} ({5 * h} min)")
    axes[0].axvline(25, color="grey", ls=":", lw=1)
    axes[0].set(xscale="log", xlabel="sources kept per sensor (K)", ylabel="proxy validation RMSE",
                title="K sweep (dotted line: K = 25, E5)")
    axes[0].legend(frameon=False)
if os.path.exists(f"{OUT}/roles.csv"):
    r = pd.read_csv(f"{OUT}/roles.csv")
    width = 0.8 / len(HORIZONS)
    for i, h in enumerate(HORIZONS):
        axes[1].barh([y + i * width for y in range(len(r))], r[f"val_rmse_h{h}"], height=width, label=f"h = {h}")
    axes[1].set_yticks([y + 0.4 - width / 2 for y in range(len(r))]); axes[1].set_yticklabels(r["setting"], fontsize=7)
    axes[1].invert_yaxis(); axes[1].set(xlabel="proxy validation RMSE", title="What B0 and P add (K = 25)")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUT}/e5_tuning_figure.png", dpi=150)
plt.show()

## 6. How to read the result

* **Calibration FAIL**: stop. The proxy does not rank graphs like A3T-GCN; tune with short A3T-GCN runs instead.
* **K sweep**: choose the K where the proxy RMSE stops improving. The proxy has no seed noise, but it is only a
  guide: confirm the best K with one A3T-GCN run against E5 (K = 25) and E1.
* **Roles**: if `VAR only (B0~ = 0)` scores as well as `P * B0 (E5)` at every horizon, LiNGAM and PEM add nothing to
  E5 and re-running them with other settings cannot help; if the rows differ, the PEM / LiNGAM runs of Phase B are worth their Kaggle hours.
* **Extras**: `top_k_overlap_with_base` near 1.0 means the new setting hardly changes the graph.

## 7. Clean up

In [ ]:
shutil.rmtree(CODE)
shutil.rmtree(GRAPHS, ignore_errors=True)
print("Output contents:", sorted(os.listdir(WORK)), "| e5_tuning_output:", sorted(os.listdir(OUT)))